# Análisis exploratorio de datos (EDA) · Saber 11 en el Valle del Cauca
**Antes (2019), durante (2020) y después (2025) de la pandemia**

Este cuaderno responde las preguntas que se hacen **antes** de analizar: *¿qué datos tenemos, qué tan buenos son y
cómo se distribuyen?* Se hace sobre el **Data Warehouse** (`saber11_dw_e2`), es decir, sobre los datos que ya pasaron
la limpieza compartida, la validación con *Great Expectations* y la carga.

| Sección | Contenido |
|---|---|
| 1 | Alcance y fuentes |
| 2 | Estructura y tipos de datos |
| 3 | Calidad: faltantes, duplicados, categorías y rangos |
| 4 | Distribuciones numéricas y valores atípicos |
| 5 | Distribuciones categóricas |
| 6 | Relaciones entre variables |
| 7 | Geografía |
| 8 | Hallazgos y decisiones |

Las pruebas de hipótesis (ANOVA, t de Welch, tamaños de efecto) están en el cuaderno
`informe_saber11.ipynb`; este EDA es la base que las justifica. Las cifras del texto se calculan al ejecutar.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

from analisis_utils import *

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.max_columns", 40, "display.width", 160)

con = conectar()
df = cargar_dw_eda(con)
ANIOS = sorted(int(a) for a in df["anio"].dropna().unique())
PAL = {str(a): COLORES_ANIO.get(a) for a in ANIOS}
ETAPA = df.drop_duplicates("anio").set_index("anio")["etapa_pandemia"].to_dict()
df["anio_txt"] = df["anio"].astype(str)
df["semestre_txt"] = df["semestre"].map({1: "Semestre 1", 2: "Semestre 2"})

PUNT = ["punt_global", *AREAS]
CATEG = ["estrato", "internet", "computador", "educ_padre", "educ_madre", "personas_hogar",
         "cole_naturaleza", "cole_calendario", "cole_area_ubicacion"]
display(Markdown(f"**{num(len(df), 0)} estudiantes** · {len(df.columns)} columnas · años {', '.join(map(str, ANIOS))}"))

## 1. Alcance y fuentes
El alcance es el **Valle del Cauca**. Cada año viene de una fuente distinta, y las tres se llevan al mismo esquema
de 25 columnas con la misma función de limpieza (`etl/limpieza.py`) antes de cargarse al DW:

| Año | Etapa | Fuente | Tarea del DAG |
|---|---|---|---|
| 2019 | Antes | CSV de datos abiertos del ICFES | `read_csv` → `transform_csv` |
| 2020 | Durante | API de datos.gov.co (Socrata) | `extract_api` → `transform_api` |
| 2025 | Después | Base de datos MySQL (`raw_saber11`) | `read_db` → `transform_db` |

El detalle de cada fuente (filas leídas, descartes, reparaciones de texto) está en `docs/ficha_fuente_*.md`.

In [ ]:
resumen = (df.groupby(["anio", "etapa_pandemia", "periodo"], observed=True)
             .size().rename("estudiantes").reset_index())
display(resumen)
display(Markdown("**Filas por tabla del Data Warehouse (esquema estrella)**"))
display(conteo_tablas(con))

# Control de la carga: filas por año frente a lo que reportan las fichas y el README (secciones 4 y 5)
ESPERADO = {2019: 57341, 2020: 41743, 2025: 50875}
ctrl = df.groupby("anio").size().rename("filas_en_el_DW").to_frame()
ctrl["filas_esperadas"] = [ESPERADO.get(int(a)) for a in ctrl.index]
ctrl["coincide"] = [("sí" if e is None or e == r else "NO") for r, e in zip(ctrl["filas_en_el_DW"], ctrl["filas_esperadas"])]
display(Markdown("**Control de la carga: filas del DW frente a las esperadas** (si alguna dice NO, revisar antes de entregar)"))
display(ctrl)

## 2. Estructura y tipos de datos

In [ ]:
tipos = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "no_nulos": df.notna().sum(),
    "nulos": df.isna().sum(),
    "valores_distintos": df.nunique(dropna=True),
})
display(tipos)
mem = df.memory_usage(deep=True).sum() / 1e6
display(Markdown(f"Memoria aproximada del conjunto: **{num(mem, 1)} MB**."))

**Lectura.** Los puntajes y el INSE son numéricos; el resto son categorías de texto. Las llaves y los códigos
(`id_resultado`, `bk_estu_consecutivo`, `cole_cod_icfes`) son identificadores: no se promedian ni se grafican.
`anio` y `semestre` se calculan en la limpieza a partir del periodo (formato `AAAAS`).

## 3. Calidad de los datos
### 3.1 Valores faltantes
Hay dos tipos de «sin dato»: el **nulo real** y la etiqueta **«No informa»** (así se codifican los faltantes de texto
en la limpieza; en el NSE se usa `-1`). «No aplica» se cuenta aparte porque es un valor estructural, no un faltante
(por ejemplo, el estudiante con inscripción individual no tiene colegio).

In [ ]:
cols_calidad = [*PUNT, "inse", "nse", "estrato", "internet", "computador", "educ_padre", "educ_madre",
                "personas_hogar", "cole_naturaleza", "cole_calendario", "cole_area_ubicacion", "municipio"]
calidad = sin_dato_por_anio(df, cols_calidad)

mapa = calidad.pivot(index="columna", columns="anio", values="sin_dato").reindex(cols_calidad)
fig, ax = plt.subplots(figsize=(7, 7))
sns.heatmap(mapa, annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": "% sin dato"}, ax=ax)
ax.set_title("% de estudiantes sin dato (nulo o «No informa») por columna y año")
ax.set_xlabel("Año"); ax.set_ylabel("")
guardar_fig(fig, "eda_faltantes")
plt.show()

detalle = calidad.pivot(index="columna", columns="anio", values=["nulo", "no_informa", "no_aplica"]).round(1)
display(Markdown("**Detalle (%)**"))
display(detalle.reindex(cols_calidad))

In [ ]:
top = (calidad.sort_values("sin_dato", ascending=False).head(5)
              .apply(lambda r: f"{r['columna']} en {int(r['anio'])} ({num(r['sin_dato'], 1)} %)", axis=1).tolist())
n_cols_ok = int((calidad.groupby("columna")["sin_dato"].max() == 0).sum())
display(Markdown("**Lectura.** Los mayores porcentajes sin dato son: " + "; ".join(top) +
                 f". {n_cols_ok} de {len(cols_calidad)} columnas no tienen faltantes en ningún año."))

### 3.2 Duplicados
La llave del estudiante (`bk_estu_consecutivo`) es única por restricción en el DW. Se verifica, y se mira además
cuántas filas tienen **idénticos** todos los atributos (eso no implica un duplicado: dos estudiantes distintos pueden
coincidir en puntajes y perfil).

In [ ]:
dup_bk = int(df["bk_estu_consecutivo"].duplicated().sum())
cols_attr = [c for c in df.columns if c not in ("id_resultado", "bk_estu_consecutivo", "anio_txt", "semestre_txt")]
dup_attr = df.duplicated(subset=cols_attr).groupby(df["anio"]).sum().rename("filas_con_atributos_idénticos")
sin_colegio = (df["cole_naturaleza"] == "No aplica").groupby(df["anio"]).sum().rename("inscripción_individual")
display(Markdown(f"**Llaves de estudiante repetidas: {dup_bk}** (debe ser 0)."))
display(pd.concat([dup_attr, sin_colegio], axis=1))

### 3.3 Consistencia de las categorías entre años
Como cada año viene de una fuente distinta, una misma categoría puede escribirse distinto. Se buscan las categorías
que aparecen en unos años y no en otros.

In [ ]:
no_comunes = categorias_no_comunes(df, CATEG)
if no_comunes.empty:
    display(Markdown("**Todas las categorías aparecen en los tres años.** No hay cambios de redacción entre fuentes."))
else:
    display(Markdown(f"**{len(no_comunes)} categorías no aparecen en todos los años:**"))
    display(no_comunes)

card = df.groupby("anio", observed=True)[CATEG].nunique().T
card.index.name = "columna"
display(Markdown("**Número de categorías distintas por año**"))
display(card)

### 3.4 Rangos válidos
Los puntajes globales van de 0 a 500 y los puntajes por área de 0 a 100. Se cuenta cualquier valor fuera de rango.

In [ ]:
limites = {"punt_global": (0, 500), **{c: (0, 100) for c in AREAS}}
filas = []
for c, (lo, hi) in limites.items():
    x = df[c].dropna()
    filas.append({"variable": c, "mínimo": x.min(), "máximo": x.max(), "fuera_de_rango": int(((x < lo) | (x > hi)).sum()),
                  "iguales_a_0": int((x == 0).sum())})
rangos = pd.DataFrame(filas)
display(rangos)
display(Markdown(f"**Valores fuera de rango: {int(rangos['fuera_de_rango'].sum())}.**"))

## 4. Distribuciones numéricas y valores atípicos

In [ ]:
desc = (df.groupby("anio", observed=True)[PUNT].describe().T.round(1))
display(Markdown("**Estadísticos descriptivos por año**"))
display(desc)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.histplot(data=df, x="punt_global", hue="anio_txt", palette=PAL, stat="density", common_norm=False,
             element="step", fill=False, bins=40, ax=axes[0])
axes[0].set_title("Distribución del puntaje global por año")
axes[0].set_xlabel("Puntaje global")
sns.boxplot(data=df, x="semestre_txt", y="punt_global", hue="anio_txt", palette=PAL, fliersize=1, ax=axes[1],
            order=["Semestre 1", "Semestre 2"])
axes[1].set_title("Puntaje global por semestre y año")
axes[1].set_xlabel(""); axes[1].set_ylabel("Puntaje global")
for a in axes:
    leg = a.get_legend()
    if leg is not None:
        leg.set_title("Año")
guardar_fig(fig, "eda_distribucion_global")
plt.show()

In [ ]:
asim = df.groupby("anio", observed=True)["punt_global"].agg(["skew", pd.Series.kurt]).round(2)
asim.columns = ["asimetría", "curtosis"]
med_sem = df.groupby("semestre", observed=True)["punt_global"].mean()
display(asim)
display(Markdown(
    f"**Lectura.** Los puntajes se concentran en una campana con la cola un poco hacia la derecha (asimetría entre "
    f"{num(asim['asimetría'].min(), 2)} y {num(asim['asimetría'].max(), 2)}). La media del semestre 1 es "
    f"{num(med_sem.get(1, float('nan')), 1)} y la del semestre 2 {num(med_sem.get(2, float('nan')), 1)}: "
    "los semestres son grupos muy distintos, por eso en el informe se comparan años dentro del mismo semestre."))

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(14, 3.6), sharey=True)
for ax, (col, nombre) in zip(axes, AREAS.items()):
    sns.boxplot(data=df, x="anio_txt", y=col, palette=PAL, hue="anio_txt", legend=False, fliersize=1, ax=ax)
    ax.set_title(nombre, fontsize=10); ax.set_xlabel(""); ax.set_ylabel("Puntaje (0–100)" if ax is axes[0] else "")
fig.suptitle("Puntaje por área y año", y=1.02)
guardar_fig(fig, "eda_areas")
plt.show()

### Valores atípicos
Se usa la regla de Tukey (fuera de Q1 − 1,5·RIC o Q3 + 1,5·RIC). Como los puntajes están acotados (0–500 y 0–100) y
no hay valores fuera de rango, un atípico es un resultado **muy alto o muy bajo pero posible**, no un error de captura.

In [ ]:
filas = []
for col in PUNT:
    for anio, g in df.groupby("anio", observed=True):
        r = outliers_iqr(g[col])
        filas.append({"variable": col, "anio": int(anio), "atípicos": r["atipicos"], "% atípicos": round(r["pct"], 2),
                      "límite inf.": round(r["lim_inf"], 1), "límite sup.": round(r["lim_sup"], 1)})
atip = pd.DataFrame(filas)
display(atip.pivot(index="variable", columns="anio", values="% atípicos").reindex(PUNT))
pct_glob = atip[atip["variable"] == "punt_global"]["% atípicos"].max()
pct_max = atip["% atípicos"].max()
display(Markdown(
    f"**Lectura.** En el puntaje global, como máximo el {num(pct_glob, 2)} % de los estudiantes de un año queda fuera de los "
    f"límites; en cualquier área, como máximo el {num(pct_max, 2)} %. **Decisión:** no se eliminan, porque son resultados "
    "plausibles dentro del rango válido y quitarlos sesgaría la comparación."))

## 5. Distribuciones categóricas

In [ ]:
vars_cat = [("estrato", "Estrato"), ("internet", "Internet en el hogar"), ("computador", "Computador en el hogar"),
            ("cole_naturaleza", "Naturaleza del colegio"), ("cole_calendario", "Calendario del colegio"),
            ("cole_area_ubicacion", "Zona del colegio")]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (col, nombre) in zip(axes.ravel(), vars_cat):
    pct = (pd.crosstab(df[col], df["anio_txt"], normalize="columns") * 100)
    pct.plot(kind="bar", ax=ax, color=[PAL.get(c) for c in pct.columns], width=0.8)
    ax.set_title(nombre); ax.set_xlabel(""); ax.set_ylabel("% de estudiantes")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(title="Año", frameon=False, fontsize=8)
fig.tight_layout()
guardar_fig(fig, "eda_categoricas")
plt.show()

In [ ]:
for col in ("educ_padre", "educ_madre"):
    pct = (pd.crosstab(df[col], df["anio"], normalize="columns") * 100).round(1)
    pct = pct.sort_values(pct.columns[-1], ascending=False).head(8)
    display(Markdown(f"**{col}: principales categorías (% por año)**"))
    display(pct)

In [ ]:
est = (pd.crosstab(df["estrato"], df["anio"], normalize="columns") * 100)
bajo = est.loc[[e for e in est.index if e in ("Estrato 1", "Estrato 2")]].sum()
display(Markdown(
    "**Lectura.** El estrato 1 y el 2 juntos son " + ", ".join(f"{num(v, 1)} % en {a}" for a, v in bajo.items()) +
    " de los estudiantes. La composición por estrato, acceso a internet y tipo de colegio **cambia entre años**, "
    "así que las comparaciones entre años deben leerse con esa composición en mente."))

## 6. Relaciones entre variables
Correlaciones de Pearson entre los puntajes y los indicadores socioeconómicos numéricos (INSE continuo y NSE de 1 a 4;
el valor `-1` del NSE significa «sin dato» y se excluye).

In [ ]:
num_df = df[[*PUNT, "inse", "nse"]].copy()
num_df.loc[num_df["nse"] == -1, "nse"] = np.nan
corr = num_df.corr().round(2)
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlación entre puntajes e indicadores socioeconómicos")
guardar_fig(fig, "eda_correlaciones")
plt.show()

c_areas = corr.loc["punt_global", list(AREAS)].sort_values(ascending=False)
c_nse = corr.loc["punt_global", ["inse", "nse"]]
display(Markdown(
    f"**Lectura.** El puntaje global se relaciona más con {AREAS[c_areas.index[0]]} (r = {num(c_areas.iloc[0], 2)}) y menos con "
    f"{AREAS[c_areas.index[-1]]} (r = {num(c_areas.iloc[-1], 2)}). Con el INSE r = {num(c_nse['inse'], 2)} y con el NSE "
    f"r = {num(c_nse['nse'], 2)}. Son asociaciones, no causas."))

### 6.1 Composición por semestre
Los dos semestres no tienen el mismo tipo de colegio. Si la composición cambia, comparar años mezclando semestres confunde el
efecto del año con el del tipo de colegio; por eso el informe compara dentro del mismo semestre.

In [ ]:
por_sem = (df.groupby(["anio", "periodo"], observed=True)
             .agg(estudiantes=("punt_global", "size"), puntaje_global_medio=("punt_global", "mean")).round(1))
display(por_sem)
for col, tit in [("cole_calendario", "Calendario del colegio"), ("cole_naturaleza", "Naturaleza del colegio")]:
    t = (pd.crosstab([df["anio"], df["periodo"]], df[col], normalize="index") * 100).round(1)
    display(Markdown(f"**{tit} por periodo (% de estudiantes)**"))
    display(t)

## 7. Geografía

In [ ]:
mun = (df.groupby("municipio", observed=True)
         .agg(estudiantes=("punt_global", "size"), puntaje_medio=("punt_global", "mean"))
         .sort_values("estudiantes", ascending=False))
top = mun.head(10).copy()
top["% del total"] = (100 * top["estudiantes"] / len(df)).round(1)
display(top.round(1))

fig, ax = plt.subplots(figsize=(8, 4))
top["puntaje_medio"][::-1].plot(kind="barh", ax=ax, color="#2a6fdb")
ax.set_xlabel("Puntaje global medio"); ax.set_ylabel("")
ax.set_title("Puntaje global medio de los 10 municipios con más estudiantes")
ax.set_xlim(left=max(0, top["puntaje_medio"].min() - 20))
guardar_fig(fig, "eda_municipios")
plt.show()
display(Markdown(
    f"**Lectura.** Hay {num(df['municipio'].nunique(), 0)} municipios; los 10 mayores reúnen el "
    f"{num(top['% del total'].sum(), 1)} % de los estudiantes. El eje del gráfico no parte de cero para que se vea la diferencia."))

## 8. Hallazgos y decisiones

In [ ]:
dup_total = int(df["bk_estu_consecutivo"].duplicated().sum())
fuera = int(rangos["fuera_de_rango"].sum())
peor = calidad.sort_values("sin_dato", ascending=False).iloc[0]
cols_con_falt = sorted(calidad.loc[calidad["sin_dato"] > 0, "columna"].unique())
h = [
    f"**Cobertura.** {num(len(df), 0)} estudiantes del Valle del Cauca en {len(ANIOS)} años "
    "(" + ", ".join(str(a) + ": " + num((df["anio"] == a).sum(), 0) for a in ANIOS) + ").",
    f"**Duplicados.** {dup_total} llaves de estudiante repetidas (la llave es única en el DW).",
    f"**Rangos.** {fuera} valores fuera de rango; los puntajes están dentro de 0–500 (global) y 0–100 (áreas).",
    f"**Faltantes.** {len(cols_con_falt)} de {len(cols_calidad)} columnas tienen algún dato faltante o «No informa»; "
    f"el caso más alto es {peor['columna']} en {int(peor['anio'])} ({num(peor['sin_dato'], 1)} %). "
    "En los análisis, los «No informa» se excluyen de cada comparación y se reporta qué porcentaje se excluyó.",
    ("**Categorías.** Todas aparecen en los tres años." if no_comunes.empty else
     f"**Categorías.** {len(no_comunes)} categorías no aparecen en todos los años (ver 3.3); hay que tenerlo en cuenta al comparar."),
    f"**Atípicos.** Como máximo {num(pct_glob, 2)} % en el puntaje global; son resultados posibles y se conservan.",
    "**Composición.** El mix de estratos, acceso a internet y tipo de colegio cambia entre años, y los dos semestres "
    "son muy distintos: por eso el informe compara siempre dentro del mismo semestre y reporta tamaños de efecto.",
]
display(Markdown("\n\n".join(f"- {x}" for x in h)))

### Decisiones que guían la transformación
Lo que muestra este EDA se traduce en estas decisiones del pipeline (README, secciones 4 y 5):

| Lo que se observa | Decisión en la transformación |
|---|---|
| Los faltantes de texto llegan de formas distintas (`nan`, `None`, vacío) | `limpiar()` los unifica en `No informa` |
| Estudiantes sin colegio (inscripción individual) | Código `-1`, nombre fijo y `No aplica` en naturaleza, calendario y área |
| NSE e INSE faltan en parte de los registros | No se imputan: NSE en `-1` e INSE en nulo; en los análisis se excluyen y se reporta la cobertura |
| El conjunto 2020-2 de la API trae el periodo `20204` | Se reasigna a `20202` antes de la limpieza compartida |
| El área del colegio llega como `URBANO` o `URBANA` | `limpiar()` homologa a `URBANA` |
| Los dos semestres tienen colegios muy distintos | Se conserva `semestre` y se compara dentro del mismo semestre |
| Un puntaje fuera de rango sería un error del dato | `quality_check` exige 0–500 el global y 0–100 cada prueba |
| Los atípicos son resultados posibles | No se eliminan |

### Limitaciones
* El EDA describe los datos **ya cargados**; los descartes previos (llaves nulas, doble inscripción, duplicados exactos) están en las fichas de cada fuente.
* La API de 2020 no trae estudiantes con inscripción individual (sin colegio), que sí están en 2019 y 2025; su segundo periodo viene como `20204` y se agrupó con `20202`.
* Las correlaciones y las diferencias entre grupos son asociaciones; las pruebas formales están en `informe_saber11.ipynb`.

### Cómo reproducir
1. Levantar el proyecto y correr el DAG `saber11_valle_etl` (el DW debe tener los tres años).
2. Generar el HTML:
   ```
   docker compose --profile analisis run --rm jupyter jupyter nbconvert --to html --execute --no-input eda_saber11.ipynb --output-dir /work/reports --output eda_saber11
   ```
3. O abrirlo de forma interactiva: `docker compose --profile analisis up -d jupyter` y entrar a http://localhost:8888 (token `saber11`).

El resultado queda en `reports/eda_saber11.html` y las figuras en `reports/figuras/`.